In [ ]:
import os
import json
from dotenv import load_dotenv
from supabase import create_client, Client

# 1. 환경 변수 및 클라이언트 설정
load_dotenv()
url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(url, key)

def run_supabase_integrated_verification_v2():
    # 1. Pending 상태의 OCR 데이터 가져오기
    pending_query = supabase.table("raw_ocr_data").select("*").eq("processing_status", "Pending").execute()
    pending_data = pending_query.data
    
    if not pending_data:
        print("💡 처리할 새로운 소명 자료가 없습니다.")
        return

    print(f"🚀 총 {len(pending_data)}건의 증빙 검증 및 % 오차 계산을 시작합니다.")

    for row in pending_data:
        r_id = row['id']
        f_name = row['file_name']
        parsed = row['raw_content'] if isinstance(row['raw_content'], dict) else json.loads(row['raw_content'])
        
        customer_no = parsed['customer_number']
        reporting_date = f"{parsed['year']}-{parsed['month']}-01"
        ocr_val = float(parsed['usage'])

        # [Step A] 매핑 정보 조회
        map_query = supabase.table("site_metric_map").select("*").eq("customer_number", customer_no).execute()
        if not map_query.data: continue
        
        mapping = map_query.data[0]
        site_id, m_name, m_unit = mapping['site_id'], mapping['metric_name'], mapping['unit']

        # [Step B] evidence_usage 적재
        evidence_res = supabase.table("evidence_usage").insert({
            "site_id": site_id, "reporting_date": reporting_date,
            "metric_name": m_name, "unit": m_unit,
            "ocr_value": ocr_val, "file_name": f_name
        }).execute()
        evidence_id = evidence_res.data[0]['id']

        # [Step C] 실적 대조 및 gap_percent 계산
        std_query = supabase.table("standard_usage").select("*")\
            .eq("site_id", site_id).eq("metric_name", m_name)\
            .eq("reporting_date", reporting_date).execute()

        if std_query.data:
            std_record = std_query.data[0]
            std_id, db_val = std_record['id'], float(std_record['value'])
            
            gap_value = db_val - ocr_val
            # [핵심] gap_percent 계산 (분모가 0인 경우 대비)
            gap_percent = (abs(gap_value) / db_val * 100) if db_val != 0 else 0

            # [Step D] 정합성 판정
            if abs(gap_value) < 1.0:
                final_status, diag = 5, "✅ 정합성 확인 완료"
            elif abs(db_val * 1000 - ocr_val) < 1.0 or abs(db_val / 1000 - ocr_val) < 1.0:
                final_status, diag = 4, "⚠️ 단위 기입 오류"
            else:
                final_status, diag = 3, f"❌ 수치 불일치 (오차: {gap_percent:.2f}%)"

            # 결과 업데이트
            supabase.table("standard_usage").update({"v_status": final_status}).eq("id", std_id).execute()
            
            # [수정] gap_percent를 포함하여 로그 기록
            log_data = {
                "std_id": std_id,
                "evidence_id": evidence_id,
                "gap_value": float(gap_value),
                "gap_percent": float(gap_percent), # 새롭게 추가된 컬럼
                "result_code": final_status,
                "diagnosis": diag
            }
            supabase.table("verification_logs").insert(log_data).execute()
            
            print(f"[{site_id}] {reporting_date} -> {diag} (Gap: {gap_percent:.2f}%)")

        # 처리 완료 업데이트
        supabase.table("raw_ocr_data").update({"processing_status": "Success"}).eq("id", r_id).execute()

if __name__ == "__main__":
    run_supabase_integrated_verification_v2()